In [ ]:

# =====================================================================
# NOTEBOOK 2 — SETUP: reload the Stage-10 full steady-state model
# =====================================================================
# This reconstructs every variable that Section 11 onward needs, by
# reading back the MODFLOW 6 files Notebook 1 already wrote to disk
# for the "ss_full" model. No re-running of Stages 2-10 required.

import os
import shutil
import numpy as np
import pandas as pd
import flopy
import matplotlib.pyplot as plt


In [ ]:
cwd = os.getcwd()
fig_dir = os.path.join(cwd, "../figures")
model_dirs = os.path.join(cwd, "../model")


In [ ]:
ss = np.array([1e-5, 5e-5, 3e-5])     # emmagasinement spécifique, 1/m
sy = np.array([0.15, 0.08, 0.05])     # porosité de drainage (seule la couche 1, libre, compte vraiment)

In [ ]:


# ---------------------------------------------------------------------------
# Grid / project constants -- must match Notebook 1 exactly
# ---------------------------------------------------------------------------
nlay, nrow, ncol = 3, 150, 100
delr, delc = 100.0, 100.0
Lx, Ly = ncol * delr, nrow * delc
PROJECT_ROOT = os.path.join(cwd, "..")

# Recreate the project directory structure (notebook 2 will write new
# transient/scenario model files into these folders)
SUBDIRS = ["model/ss_basic", "model/ss_full", "model/transient_baseline",
           "model/transient_reduced_recharge", "model/transient_increased_pumping",
           "figures", "data", "output"]
for d in SUBDIRS:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
os.makedirs("figures", exist_ok=True)

# ---------------------------------------------------------------------------
# Locate the MODFLOW 6 executable (same helper as Notebook 1)
# ---------------------------------------------------------------------------
def find_mf6_executable():
    candidates = [shutil.which("mf6"), "../bin/mf6", "bin/mf6",
                  os.path.expanduser("~/.local/bin/mf6")]
    for c in candidates:
        if c and os.path.isfile(c) and os.access(c, os.X_OK):
            return os.path.abspath(c)
    return None

MF6_EXE = find_mf6_executable()
if MF6_EXE is None:
    flopy.utils.get_modflow("./bin", subset="mf6")
    MF6_EXE = find_mf6_executable()
assert MF6_EXE is not None, "Set MF6_EXE manually if it still can't be found."

# ---------------------------------------------------------------------------
# Reload the full steady-state simulation ("ss_full") written by Notebook 1
# ---------------------------------------------------------------------------
ss_full_ws = os.path.join(PROJECT_ROOT, "model/ss_full")
sim_full = flopy.mf6.MFSimulation.load(sim_ws=ss_full_ws, exe_name=exe_name, verbosity_level=0)
gwf_full = sim_full.get_model("ss_full")

heads_full = gwf_full.output.head().get_data()

# ---------------------------------------------------------------------------
# Recover grid geometry
# ---------------------------------------------------------------------------
top  = gwf_full.dis.top.array
botm = gwf_full.dis.botm.array

# ---------------------------------------------------------------------------
# Recover heterogeneous hydraulic properties (Stage 4)
# ---------------------------------------------------------------------------
kx = np.array(gwf_full.npf.k.array)
kz = np.array(gwf_full.npf.k33.array)
icelltype = list(gwf_full.npf.icelltype.array)

# NOTE: ss/sy are NOT recoverable from ss_full -- the steady-state model has
# no STO package (storage only matters once transient). These were simple
# fixed per-layer constants in Notebook 1, so we just restate them here.
# If you ever change these in Notebook 1, mirror the change here too.
ss = np.array([1e-5, 5e-5, 3e-5])     # specific storage, 1/m
sy = np.array([0.15, 0.08, 0.05])     # specific yield

# ---------------------------------------------------------------------------
# Recover recharge (Stage 8) and EVT (Stage 7)
# ---------------------------------------------------------------------------
rch_arr = np.array(gwf_full.rcha.recharge.array)
rch_spatial = rch_arr[0] if rch_arr.ndim == 3 else rch_arr

evt_surface_arr = np.array(gwf_full.evta.surface.array)
evt_surface = evt_surface_arr[0] if evt_surface_arr.ndim == 3 else evt_surface_arr
evt_rate_arr = np.array(gwf_full.evta.rate.array)
evt_rate_uniform = float(np.unique(evt_rate_arr)[0])
evt_depth_arr = np.array(gwf_full.evta.depth.array)
evt_extdp = float(np.unique(evt_depth_arr)[0])

# ---------------------------------------------------------------------------
# Recover CHD (north/south boundaries, Stage 2/10)
# ---------------------------------------------------------------------------
chd_rec = gwf_full.chd.stress_period_data.get_data()[0]
chd_spd_ns_only = [(int(cid[0]), int(cid[1]), int(cid[2]), float(h))
                    for cid, h in zip(chd_rec["cellid"], chd_rec["head"])]

# ---------------------------------------------------------------------------
# Recover the river (Stage 5) and rebuild the row->column / stage / bed arrays
# ---------------------------------------------------------------------------
riv_rec = gwf_full.riv.stress_period_data.get_data()[0]
riv_spd = [(int(cid[0]), int(cid[1]), int(cid[2]), float(stage), float(cond), float(rbot))
           for cid, stage, cond, rbot in zip(riv_rec["cellid"], riv_rec["stage"],
                                              riv_rec["cond"], riv_rec["rbot"])]

river_col = np.zeros(nrow, dtype=int)
river_stage = np.zeros(nrow)
river_bed_elev = np.zeros(nrow)
for lay, r, c, stage, cond, rbot in riv_spd:
    river_col[r] = c
    river_stage[r] = stage
    river_bed_elev[r] = rbot
riverbed_K = 1.0                              # synthetic constant from Notebook 1
riv_conductance = float(riv_spd[0][4])        # uniform across cells

# ---------------------------------------------------------------------------
# Recover the wells (Stage 6) and rebuild well_df exactly as before
# ---------------------------------------------------------------------------
wel_rec = gwf_full.wel.stress_period_data.get_data()[0]
wells = [[f"WEL-{i+1:02d}", int(cid[0]), int(cid[1]), int(cid[2]), float(q)]
         for i, (cid, q) in enumerate(zip(wel_rec["cellid"], wel_rec["q"]))]
well_df = pd.DataFrame(wells, columns=["well_id", "layer", "row", "col", "q_m3d"])
well_df["x"] = well_df["col"] * delr + delr/2
well_df["y"] = Ly - (well_df["row"] * delc + delc/2)
well_df["dist_to_river_m"] = [abs(well_df.col[i] - river_col[well_df.row[i]]) * delr
                                for i in well_df.index]

# ---------------------------------------------------------------------------
# Reusable helper functions Section 11+ depends on
# ---------------------------------------------------------------------------
def run_sim(sim, verbose_on_fail=True):
    sim.write_simulation(silent=True)
    success, buff = sim.run_simulation(silent=True)
    if not success and verbose_on_fail:
        print("\n".join(buff[-40:]))
    return success

def check_water_budget(gwf, kstpkper=None, tolerance_pct=1.0):
    cbc = gwf.output.budget()
    kk = kstpkper if kstpkper is not None else cbc.get_kstpkper()[-1]
    names = [n.decode().strip() if isinstance(n, bytes) else n for n in cbc.get_unique_record_names()]
    total_in, total_out = 0.0, 0.0
    for name in names:
        if name in ("FLOW-JA-FACE", "DATA-SPDIS"):
            continue
        rec = cbc.get_data(kstpkper=kk, text=name)[0]
        q = rec["q"] if rec.dtype.names is not None else rec.ravel()
        total_in += q[q > 0].sum()
        total_out += q[q < 0].sum()
    discrepancy_pct = 100 * (total_in + total_out) / max(abs(total_in), 1e-9)
    ok = abs(discrepancy_pct) < tolerance_pct
    print(f"  Total IN = {total_in:,.1f} m3/d | Total OUT = {total_out:,.1f} m3/d "
          f"| Discrepancy = {discrepancy_pct:.3f}% | {'OK' if ok else 'CHECK MODEL'}")
    return discrepancy_pct

# ---------------------------------------------------------------------------
# Sanity check -- confirms everything reloaded correctly before continuing
# ---------------------------------------------------------------------------
print("Reloaded ss_full model successfully.")
print(f"  heads_full range: {np.nanmin(heads_full):.1f} - {np.nanmax(heads_full):.1f} m")
print(f"  kx layers: {kx.shape}, wells: {len(well_df)}, river cells: {len(riv_spd)}, "
      f"CHD cells: {len(chd_spd_ns_only)}")
_ = check_water_budget(gwf_full)

---
## 11. Étape 9 — Passage à une simulation transitoire

Tout, jusqu'ici, était en **régime permanent** : un équilibre « instantané » unique qui
ignore l'évolution des conditions dans le temps. Les vrais aquifères répondent à un
cycle saisonnier de pluie, d'ETP et de pompage — pour le représenter, nous convertissons
le modèle en régime **transitoire**, en utilisant **12 périodes de stress mensuelles
représentant une année hydrologique complète.**

**Concepts clés du régime transitoire :**

- **Période de stress** — un intervalle pendant lequel toutes les sollicitations aux
  limites (recharge, pompage, etc.) restent constantes. Nous utilisons une période de
  stress par mois calendaire.
- **Pas de temps** — les périodes de stress sont subdivisées en pas de temps pour la
  précision numérique ; nous utilisons 6 pas de temps par mois avec un multiplicateur
  de 1,3× (les pas grandissent légèrement au cours du mois, une pratique standard qui
  améliore la convergence juste après un changement de sollicitation).
- **Régime permanent vs transitoire** — une période en régime permanent ignore
  totalement les termes d'emmagasinement ; une période transitoire nécessite le package
  `STO` pour que MODFLOW 6 puisse suivre l'eau qui entre ou sort de l'emmagasinement à
  mesure que les charges montent et descendent.
- **Emmagasinement spécifique (`Ss`)** — le volume d'eau qu'un aquifère *captif*
  libère par unité de volume et par unité de baisse de charge (1/m). Très petit
  (10⁻⁵–10⁻⁴ /m) car il ne reflète que la compressibilité de l'eau et du squelette de
  l'aquifère.
- **Porosité de drainage (`Sy`)** — le volume d'eau qu'un aquifère *libre* libère par
  unité de baisse de charge lorsque la nappe draine physiquement l'espace poreux.
  Beaucoup plus grand (0,05–0,25) car il reflète un véritable dénoyage, pas seulement
  une compression.
- La couche 1 (libre, `icelltype=1`) utilise `Sy` ; les couches 2–3 (captives,
  `icelltype=0`) n'utilisent que `Ss`.

Les propriétés d'emmagasinement ont déjà été définies à l'étape 4 (tableaux `ss`, `sy`) ;
nous les réutilisons ici.

In [ ]:
print("Emmagasinement spécifique (Ss, 1/m) par couche :", ss)
print("Porosité de drainage (Sy, -) par couche :        ", sy)
print("icelltype (1=convertible/libre, 0=captif) :", icelltype)

---
## 12. Étape 10 — Jeu de données climatiques synthétiques du sud du Burkina Faso

**Jeu de données climatiques synthétiques du sud du Burkina Faso, à des fins de
formation.** Il s'agit de valeurs mensuelles *hypothétiques* choisies pour reproduire un
schéma saisonnier soudano-sahélien reconnaissable — une longue saison sèche (environ
novembre–février), une transition chaude pré-pluvieuse (mars–mai), une saison des
pluies (juin–septembre), et une transition de retour vers des conditions sèches
(octobre). Ce ne sont **pas** des relevés mesurés d'une station météorologique
particulière.

In [ ]:
months = ["Jan","Fév","Mar","Avr","Mai","Juin","Juil","Août","Sep","Oct","Nov","Déc"]
days_in_month = np.array([31,28,31,30,31,30,31,31,30,31,30,31])   # année de référence (non bissextile)

precip_mm = np.array([0, 2, 10, 35, 90, 140, 220, 270, 190, 60, 5, 0], dtype=float)

# La recharge répond de façon non linéaire à la pluie : peu d'infiltration sous un
# seuil d'humidité du sol, puis une fraction à peu près constante de la pluie
# au-dessus de ce seuil (hypothèse synthétique).
recharge_threshold_mm = 20
recharge_efficiency = 0.14
recharge_mm = np.clip(precip_mm - recharge_threshold_mm, 0, None) * recharge_efficiency

# ETP potentielle : maximale pendant la saison chaude pré-pluvieuse, plus faible
# pendant la saison sèche fraîche et pendant la saison des pluies plus nuageuse.
pet_mm = np.array([150, 160, 205, 210, 195, 150, 130, 120, 140, 165, 150, 140], dtype=float)
et_rate_fraction_of_pet = 0.55   # coefficient cultural/végétation appliqué pour obtenir le taux EVT max de MODFLOW

# La demande de pompage augmente en saison sèche (irrigation + demande domestique plus
# forte) et diminue pendant les pluies.
pumping_multiplier = np.array([1.35, 1.30, 1.45, 1.40, 1.15, 0.75, 0.60, 0.60, 0.70, 0.95, 1.20, 1.30])

climate_df = pd.DataFrame({
    "month": months, "days": days_in_month,
    "precip_mm": precip_mm, "recharge_mm": recharge_mm, "pet_mm": pet_mm,
    "recharge_m_d": recharge_mm / 1000.0 / days_in_month,                 # mm/mois -> m/jour
    "evt_rate_m_d": (pet_mm * et_rate_fraction_of_pet) / 1000.0 / days_in_month,
    "pumping_multiplier": pumping_multiplier,
})
climate_df.to_csv(os.path.join(PROJECT_ROOT, "data", "synthetic_climate_southern_burkina_faso.csv"),
                    index=False)
climate_df.round(6)

In [ ]:
annual_precip = precip_mm.sum()
annual_recharge = recharge_mm.sum()
print(f"Précipitations annuelles : {annual_precip:.0f} mm/an  (valeur synthétique de formation)")
print(f"Recharge annuelle :        {annual_recharge:.0f} mm/an")
print(f"Efficacité de recharge implicite : {100*annual_recharge/annual_precip:.1f}% des précipitations")

### Visualisation du climat


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

axes[0,0].bar(months, precip_mm, color="steelblue")
axes[0,0].set_title("Précipitations mensuelles"); axes[0,0].set_ylabel("mm")

axes[0,1].plot(months, recharge_mm, "o-", color="navy", label="Recharge")
axes[0,1].plot(months, pet_mm/10, "o--", color="darkorange", label="ETP / 10 (à l'échelle)")
axes[0,1].set_title("Recharge mensuelle vs ETP potentielle (à l'échelle)"); axes[0,1].legend(fontsize=8)

axes[1,0].scatter(precip_mm, recharge_mm, color="navy")
axes[1,0].set_xlabel("Précipitations (mm)"); axes[1,0].set_ylabel("Recharge (mm)")
axes[1,0].set_title("Précipitations vs recharge (réponse à seuil non linéaire)")

ax2 = axes[1,1]
ax2.bar(months, precip_mm, color="steelblue", alpha=0.5, label="Précipitations (mm)")
ax2b = ax2.twinx()
ax2b.plot(months, pumping_multiplier, "s-", color="firebrick", label="Multiplicateur de pompage")
ax2.set_title("Sollicitations saisonnières combinées")
ax2.set_ylabel("Précipitations (mm)"); ax2b.set_ylabel("Multiplicateur de pompage (relatif à la moyenne)")
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2b.get_legend_handles_labels()
ax2.legend(lines1+lines2, labels1+labels2, fontsize=8, loc="upper right")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("../figures/stage10_climate_forcing.pdf", dpi=150)
plt.show()


---
## 13. Étapes 11–12 — Périodes de stress mensuelles, recharge/ETP/pompage saisonniers

Nous traduisons maintenant le jeu de données climatiques en données de période de
stress MODFLOW 6 de façon programmatique — aucun tableau saisi manuellement pour
chacun des 12 mois. Les taux de recharge et d'ETP sont mis à l'échelle mois par mois à
partir des motifs spatiaux déjà construits (étapes 7–8) ; les débits de pompage sont mis
à l'échelle à partir des débits de référence des puits, en utilisant le même
multiplicateur mensuel.

In [ ]:
def build_transient_model(sim_name, sim_ws, recharge_scale=1.0, pumping_scale=1.0,
                           evt_scale=1.0, strt=None):
    """Construit un modèle MODFLOW 6 transitoire sur 12 mois. recharge_scale /
    pumping_scale / evt_scale nous permettent de construire des scénarios alternatifs
    (étape 23) en réutilisant cette seule fonction."""
    sim = flopy.mf6.MFSimulation(sim_name=sim_name, sim_ws=sim_ws, exe_name=exe_name)
    perioddata = [(float(d), 6, 1.3) for d in days_in_month]
    flopy.mf6.ModflowTdis(sim, time_units="days", nper=12, perioddata=perioddata)
    # flopy.mf6.ModflowIms(sim, complexity="COMPLEX", outer_dvclose=1e-3,
    #                       inner_dvclose=1e-4, outer_maximum=500, inner_maximum=200)

    flopy.mf6.ModflowIms(
        sim, complexity="COMPLEX",
        outer_dvclose=1e-3, inner_dvclose=1e-4,
        outer_maximum=800, inner_maximum=300,
        backtracking_number=20, backtracking_tolerance=1.05,
        backtracking_reduction_factor=0.3, backtracking_residual_limit=1e-3,
    )

    gwf = flopy.mf6.ModflowGwf(sim, modelname=sim_name, save_flows=True,
                                newtonoptions="NEWTON UNDER_RELAXATION")
    flopy.mf6.ModflowGwfdis(gwf, nlay=nlay, nrow=nrow, ncol=ncol,
                             delr=delr, delc=delc, top=top, botm=botm)
    if strt is None:
        strt = np.stack([top.copy() - 3] * nlay)
    flopy.mf6.ModflowGwfic(gwf, strt=strt)
    flopy.mf6.ModflowGwfnpf(gwf, icelltype=icelltype, k=kx, k33=kz, save_specific_discharge=True)
    flopy.mf6.ModflowGwfsto(gwf, iconvert=icelltype, ss=ss, sy=sy,
                             steady_state={}, transient={i: True for i in range(12)})
    flopy.mf6.ModflowGwfchd(gwf, stress_period_data=chd_spd_ns_only)

    rch_norm = climate_df.recharge_m_d / max(climate_df.recharge_m_d.mean(), 1e-12)
    rch_spd = {i: rch_spatial * rch_norm.iloc[i] * recharge_scale for i in range(12)}
    flopy.mf6.ModflowGwfrcha(gwf, recharge=rch_spd)

    # evt_spd = {i: [(0, r, c, float(evt_surface[r, c]),
    #                  float(climate_df.evt_rate_m_d.iloc[i]) * evt_scale, evt_extdp)
    #                 for r in range(nrow) for c in range(ncol)]
    #            for i in range(12)}
    # flopy.mf6.ModflowGwfevt(gwf, nseg=1, stress_period_data=evt_spd)


    # Array-based EVT (consistent with ModflowGwfrcha used for recharge above).
    # surface and depth are constant across the year, so a single array/scalar
    # is enough; only rate varies month to month.
    evt_rate_spd = {i: float(climate_df.evt_rate_m_d.iloc[i]) * evt_scale
                 for i in range(12)}

    flopy.mf6.ModflowGwfevta(
    gwf,
    surface=evt_surface,   # 2D array, constant across periods
    rate=evt_rate_spd,     # dict {kper: scalar or array}, MF6 broadcasts scalars as CONSTANT
    depth=evt_extdp,       # constant scalar, applies to every period
    )


    flopy.mf6.ModflowGwfriv(gwf, stress_period_data={0: riv_spd}, save_flows=True)

    wel_spd = {i: [((int(w.layer), int(w.row), int(w.col)),
                     float(w.q_m3d) * climate_df.pumping_multiplier.iloc[i] * pumping_scale)
                    for w in well_df.itertuples()]
               for i in range(12)}
    flopy.mf6.ModflowGwfwel(gwf, stress_period_data=wel_spd, save_flows=True, auto_flow_reduce=0.1)

    flopy.mf6.ModflowGwfoc(gwf, budget_filerecord=f"{sim_name}.cbc",
                            head_filerecord=f"{sim_name}.hds",
                            saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")])
    return sim, gwf

print("build_transient_model() défini -- construction programmatique des périodes de "
      "stress sur 12 mois, aucun tableau mensuel saisi manuellement.")


---
## 14. Étape 13 — Exécuter et analyser le modèle transitoire

Nous initialisons la simulation transitoire à partir des charges du modèle complet en
régime permanent (`heads_full`) — une « condition de départ typique » raisonnable pour
le début de l'année simulée.

In [ ]:
sim_tr, gwf_tr = build_transient_model(
    "tr_baseline", os.path.join(PROJECT_ROOT, "model/transient_baseline"), strt=heads_full)

success_tr = run_sim(sim_tr)
print("Modèle transitoire de référence convergé (12 mois) :", success_tr)
_ = check_water_budget(gwf_tr)  # contrôle le dernier pas de temps par défaut

In [ ]:
monitor_points = pd.DataFrame([
    {"name": "MW-1 Près d'un puits (peu profond)",  "row": well_df.row[0],  "col": well_df.col[0],  "layer": 0,
     "reason": "Suit directement la réponse en rabattement à côté d'un puits de pompage"},
    {"name": "MW-2 Près de la rivière",              "row": 75, "col": int(river_col[75])+2, "layer": 0,
     "reason": "Suit l'interaction rivière-aquifère et l'amortissement des variations saisonnières"},
    {"name": "MW-3 Entre deux puits",                "row": 90, "col": 45, "layer": 1,
     "reason": "Capture le rabattement cumulé/d'interférence entre plusieurs puits"},
    {"name": "MW-4 Amont hydraulique (nord)",        "row": 20, "col": 50, "layer": 0,
     "reason": "Signal saisonnier de fond, influence minimale du pompage"},
    {"name": "MW-5 Aval hydraulique (sud)",          "row": 130, "col": 50, "layer": 0,
     "reason": "Montre comment le signal saisonnier/de pompage se propage vers l'exutoire"},
    {"name": "MW-6 Couche profonde",                 "row": 90, "col": 45, "layer": 2,
     "reason": "Teste si la couche profonde à faible K montre une réponse amortie/retardée"},
])
monitor_points

In [ ]:
def get_hydrograph(gwf, row, col, layer):
    hb = gwf.output.head()
    kstpkper = hb.get_kstpkper()
    times = np.array(hb.get_times())
    vals = np.array([hb.get_data(kstpkper=kk)[layer, row, col] for kk in kstpkper])
    return times, vals

def plot_hydrograph(gwf, points_df, title="Hydrogrammes simulés"):
    """Deux panneaux : (gauche) charges absolues pour chaque point de suivi -- montre
    l'altitude réelle et le rabattement total, y compris la variation de plusieurs
    mètres du puits pompé ; (droite) anomalie de charge pour les points *non pompés*
    seulement, chacun par rapport à sa propre moyenne annuelle -- zoomé sur son propre
    axe afin que le cycle saisonnier d'origine climatique, bien plus petit (échelle
    décimétrique) à ces points, soit réellement visible plutôt qu'aplati en une ligne
    plate à côté de la variation bien plus grande du puits pompé."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for _, p in points_df.iterrows():
        t, h = get_hydrograph(gwf, p.row, p.col, p.layer)
        axes[0].plot(t, h, marker="o", ms=3, label=p["name"])
        if not p["name"].startswith("MW-1"):
            axes[1].plot(t, h - h.mean(), marker="o", ms=3, label=p["name"])
    axes[0].set_xlabel("Temps (jours dans l'année simulée)"); axes[0].set_ylabel("Charge simulée (m NGF)")
    axes[0].set_title(title + " -- charge absolue (tous points)")
    axes[1].set_xlabel("Temps (jours dans l'année simulée)"); axes[1].set_ylabel("Anomalie de charge (m, par rapport à la moyenne annuelle)")
    axes[1].set_title(title + " -- anomalie saisonnière (points non pompés, zoomé)")
    axes[1].axhline(0, color="k", lw=0.6)
    axes[0].legend(fontsize=7, loc="best")
    axes[1].legend(fontsize=7, loc="best")
    return fig, axes

plot_hydrograph(gwf_tr, monitor_points)
plt.tight_layout()
plt.savefig("../figures/stage13_hydrographs.pdf", dpi=150)
plt.show()

### Réponse saisonnière de la nappe et décalage temporel

Un système d'eau souterraine ne répond pas instantanément à la pluie — l'eau doit
s'infiltrer, et le changement de charge qui en résulte doit se propager dans
l'aquifère. Ce décalage dépend de la **diffusivité hydraulique** de l'aquifère
(approximativement `K/Ss` ou `T/S` — la vitesse à laquelle une perturbation de charge
se propage) — les aquifères à K élevé et faible emmagasinement répondent le plus vite.

In [ ]:
t, h_upgradient = get_hydrograph(gwf_tr, monitor_points.loc[3,"row"], monitor_points.loc[3,"col"], 0)
month_mid_days = np.cumsum(days_in_month) - days_in_month/2

fig, ax1 = plt.subplots(figsize=(11, 4.5))
ax1.bar(month_mid_days, precip_mm, width=25, color="steelblue", alpha=0.4, label="Précipitations (mm)")
ax1.set_ylabel("Précipitations (mm/mois)")
ax2 = ax1.twinx()
ax2.plot(t, h_upgradient, "o-", color="darkred", label="Charge MW-4")
ax2.set_ylabel("Charge simulée (m NGF)")
ax1.set_xlabel("Jour de l'année simulée")
ax1.set_title("Pluie vs réponse du niveau de la nappe (illustrant une réaction décalée)")
lines1, labels1 = ax1.get_legend_handles_labels(); lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc="upper left", fontsize=8)
plt.tight_layout()
plt.savefig("../figures/stage13_lag.pdf", dpi=150)
plt.show()

peak_rain_day = month_mid_days[np.argmax(precip_mm)]
peak_head_day = t[np.argmax(h_upgradient)]
print(f"Le pic de précipitations survient vers le jour {peak_rain_day:.0f} de l'année")
print(f"Le pic de charge simulée survient vers le jour {peak_head_day:.0f} de l'année")
print(f"Décalage approximatif : {peak_head_day - peak_rain_day:.0f} jours")

---
## 15. Analyse du bilan hydrique

In [ ]:
def calculate_water_budget(gwf, package_names=("STO-SS","STO-SY","WEL","RIV","RCHA","EVT","CHD")):
    """Fonction réutilisable : construit un tableau de bilan hydrique période de
    stress par période de stress."""
    cbc = gwf.output.budget()
    kstpkper_list = cbc.get_kstpkper()
    rows = []
    for kk in kstpkper_list:
        rec = {"kstp": kk[0], "kper": kk[1]}
        for pkg in package_names:
            try:
                data = cbc.get_data(kstpkper=kk, text=pkg)[0]
                rec[pkg] = data["q"].sum() if data.dtype.names is not None else data.sum()
            except Exception:
                rec[pkg] = np.nan
        rows.append(rec)
    df = pd.DataFrame(rows)
    return df.groupby("kper", as_index=True).sum(numeric_only=True).drop(columns=["kstp"])

monthly_budget = calculate_water_budget(gwf_tr)
monthly_budget.index = months
monthly_budget.round(0)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
monthly_budget[["RCHA","EVT","WEL","RIV"]].plot(kind="bar", ax=ax,
    color=["seagreen","goldenrod","firebrick","steelblue"])
ax.axhline(0, color="k", lw=0.8)
ax.set_ylabel("Flux (m3/jour, + = entrant dans l'aquifère, - = sortant de l'aquifère)")
ax.set_title("Bilan hydrique mensuel par composante (modèle transitoire de référence)")
ax.legend(["Recharge","ETP","Puits","Rivière"])
plt.tight_layout()
plt.savefig("../figures/stage15_water_budget.pdf", dpi=150)
plt.show()

In [ ]:
annual_budget = monthly_budget.sum()
inflow_terms = ["RCHA"]
outflow_terms = ["EVT", "WEL"]
print("Totaux annuels (m3/an) :")
print(annual_budget.round(0))
print()
total_in = annual_budget[annual_budget > 0].sum()
total_out = annual_budget[annual_budget < 0].sum()
print(f"Somme des termes positifs (entrées) : {total_in:,.0f} m3/an")
print(f"Somme des termes négatifs (sorties) : {total_out:,.0f} m3/an")
print(f"Net (entrées + sorties) : {total_in + total_out:,.0f} m3/an  "
      "(proche de zéro => la limite CHD compense -- attendu, puisqu'elle représente "
      "les échanges régionaux au-delà de notre domaine de modèle)")

# contribution en pourcentage des termes principaux
pct = (annual_budget.abs() / annual_budget.abs().sum() * 100).round(1)
print("\nContribution en pourcentage de chaque terme du bilan (en amplitude) :")
print(pct)

---
## 16. Diagnostics du modèle et contrôle qualité

Un contrôle qualité professionnel va au-delà de « le solveur a-t-il signalé un
succès ». Nous vérifions la convergence, la fermeture du bilan de masse sur l'année
entière, les mailles asséchées, les gradients non réalistes, et si nos hypothèses de
conditions aux limites (connexion à la rivière, plages de propriétés hydrauliques)
tiennent toujours sous des sollicitations transitoires plus dynamiques.

In [ ]:
def qc_report(gwf, sim):
    print("="*70)
    print(f"RAPPORT DE CONTRÔLE QUALITÉ : {gwf.name}")
    print("="*70)

    # 1. Convergence
    success, buff = sim.run_simulation(silent=True)
    print(f"[1] Convergence du solveur : {'OK' if success else 'ÉCHEC'}")

    # 2. Fermeture du bilan hydrique sur toutes les périodes de stress
    cbc = gwf.output.budget()
    worst = 0.0
    for kk in cbc.get_kstpkper():
        names = [n.decode().strip() if isinstance(n, bytes) else n for n in cbc.get_unique_record_names()]
        tin = tout = 0.0
        for name in names:
            if name in ("FLOW-JA-FACE", "DATA-SPDIS"):
                continue
            rec = cbc.get_data(kstpkper=kk, text=name)[0]
            q = rec["q"] if rec.dtype.names is not None else rec.ravel()
            tin += q[q > 0].sum(); tout += q[q < 0].sum()
        disc = 100 * (tin + tout) / max(abs(tin), 1e-9)
        worst = max(worst, abs(disc))
    print(f"[2] Pire écart de bilan de masse sur l'année : {worst:.3f}%  "
          f"({'OK' if worst < 1.0 else 'À VÉRIFIER'})")

    # 3. Mailles asséchées / charges non réalistes
    hb = gwf.output.head()
    h_end = hb.get_data(kstpkper=hb.get_kstpkper()[-1])
    dry = np.isnan(h_end) | (h_end < -1e10)
    print(f"[3] Mailles asséchées ou inactives en fin de simulation : {dry.sum()}  "
          f"({'OK' if dry.sum()==0 else 'À VÉRIFIER'})")

    # 4. Charges très en dehors de la plage d'altitude plausible
    implausible = ((h_end < botm[-1].min() - 5) | (h_end > top.max() + 20))
    print(f"[4] Mailles à charge invraisemblable (hors plage d'altitude du modèle +/- marge) : "
          f"{np.nansum(implausible)}  ({'OK' if np.nansum(implausible)==0 else 'À VÉRIFIER'})")

    # 5. Contrôle de vraisemblance de la plage de conductivité hydraulique
    k_ok = (kx.min() > 0) and (kx.max() < 100)  # bornes généreuses pour ce type de roche
    print(f"[5] Conductivité hydraulique dans une plage plausible (0-100 m/jour) : "
          f"{'OK' if k_ok else 'À VÉRIFIER'}")

    # 6. Cohérence des altitudes de la rivière : le niveau doit dépasser l'altitude du lit partout
    riv_ok = np.all(river_stage > river_bed_elev)
    print(f"[6] Niveau de la rivière au-dessus de l'altitude du lit partout : "
          f"{'OK' if riv_ok else 'À VÉRIFIER'}")

    print("="*70)
    return {"converged": success, "worst_discrepancy_pct": worst, "n_dry_cells": int(dry.sum())}

_ = qc_report(gwf_tr, sim_tr)


---
## 17. Étape 23 — Analyse de scénarios

Nous comparons trois scénarios pour montrer comment un modèle d'eau souterraine devient
un **outil d'aide à la décision**, et pas seulement un outil de prédiction :

1. **Référence** — le modèle transitoire tel que construit (déjà exécuté ci-dessus).
2. **Recharge réduite** — recharge ramenée à 70% de la référence (représentant, par
   exemple, une année plus sèche que la normale).
3. **Pompage augmenté** — pompage porté à 150% de la référence (représentant, par
   exemple, une expansion agricole ou une croissance démographique).

In [ ]:
sim_red, gwf_red = build_transient_model(
    "tr_redrch", os.path.join(PROJECT_ROOT, "model/transient_reduced_recharge"),
    recharge_scale=0.70, strt=heads_full)
success_red = run_sim(sim_red)

sim_pump, gwf_pump = build_transient_model(
    "tr_incpump", os.path.join(PROJECT_ROOT, "model/transient_increased_pumping"),
    pumping_scale=1.5, strt=heads_full)
success_pump = run_sim(sim_pump)

print("Scénario recharge réduite convergé :", success_red)
print("Scénario pompage augmenté convergé :", success_pump)


In [ ]:
def compare_scenarios(scenario_models, well_df, monitor_points):
    """Fonction réutilisable : construit un tableau comparatif des scénarios en
    utilisant le dernier pas de temps (fin d'année) de chaque simulation transitoire."""
    rows = []
    for name, gwf in scenario_models.items():
        hb = gwf.output.head()
        h_end = hb.get_data(kstpkper=hb.get_kstpkper()[-1])
        cbc = gwf.output.budget()
        riv_q = cbc.get_data(kstpkper=hb.get_kstpkper()[-1], text="RIV")[0]
        riv_total = riv_q["q"].sum()
        well_heads = [h_end[int(w.layer), int(w.row), int(w.col)] for w in well_df.itertuples()]
        rows.append({
            "scenario": name,
            "mean_head_layer1": np.nanmean(h_end[0]),
            "max_drawdown_from_full_ss": np.nanmax(heads_full[0] - h_end[0]),
            "mean_well_head": np.mean(well_heads),
            "net_river_exchange_m3d": riv_total,
        })
    return pd.DataFrame(rows).set_index("scenario")

scenario_models = {"Référence": gwf_tr, "Recharge réduite (70%)": gwf_red,
                    "Pompage augmenté (150%)": gwf_pump}
scenario_comparison = compare_scenarios(scenario_models, well_df, monitor_points)
scenario_comparison.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
scenario_comparison["mean_head_layer1"].plot(kind="bar", ax=axes[0], color=["grey","goldenrod","firebrick"])
axes[0].set_ylabel("Charge moyenne, couche 1, fin d'année (m NGF)")
axes[0].set_title("Altitude moyenne de la nappe par scénario (axe y zoomé)")
axes[0].tick_params(axis="x", rotation=20)
# On zoome l'axe y sur la plage réelle des données -- le gradient régional contrôlé par
# CHD signifie que la charge moyenne *absolue* bouge à peine entre scénarios ; un axe
# démarrant à zéro aplatirait ces barres à l'identique visuellement, masquant une
# différence réelle et significative.
vals = scenario_comparison["mean_head_layer1"]
pad = max(0.05, (vals.max() - vals.min()) * 0.6)
axes[0].set_ylim(vals.min() - pad, vals.max() + pad)

scenario_comparison["net_river_exchange_m3d"].plot(kind="bar", ax=axes[1], color=["grey","goldenrod","firebrick"])
axes[1].axhline(0, color="k", lw=0.8)
axes[1].set_ylabel("Échange net avec la rivière, fin d'année (m3/jour)\n[négatif = gain net pour la rivière]")
axes[1].set_title("Impact sur la rivière par scénario")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.savefig("../figures/stage17_scenarios.pdf", dpi=150)
plt.show()

---
## 18. Exercice introductif de calage

**Toutes les « observations » ci-dessous sont synthétiques.** Nous les générons en
échantillonnant les charges simulées de notre propre modèle de référence en des
emplacements aléatoires, puis en ajoutant un bruit de mesure aléatoire réaliste — c'est
une façon courante de *s'entraîner* aux flux de travail de calage avant de travailler
avec de vraies données de terrain.

In [ ]:
rng_obs = np.random.default_rng(99)
n_obs = 25
obs_rows = rng_obs.integers(10, nrow-10, n_obs)
obs_cols = rng_obs.integers(10, ncol-10, n_obs)
obs_layers = rng_obs.integers(0, nlay, n_obs)

h_end_baseline = gwf_tr.output.head().get_data(kstpkper=gwf_tr.output.head().get_kstpkper()[-1])
true_heads = np.array([h_end_baseline[obs_layers[i], obs_rows[i], obs_cols[i]] for i in range(n_obs)])
measurement_noise = rng_obs.normal(0, 0.5, n_obs)   # +/- 0.5 m d'erreur de mesure synthétique
observed_heads = true_heads + measurement_noise

obs_df = pd.DataFrame({
    "obs_id": [f"OBS-{i+1:02d}" for i in range(n_obs)],
    "row": obs_rows, "col": obs_cols, "layer": obs_layers,
    "simulated_m": true_heads, "observed_m (SYNTHÉTIQUE)": observed_heads,
})
obs_df["residual_m"] = obs_df["observed_m (SYNTHÉTIQUE)"] - obs_df["simulated_m"]
obs_df.round(3)

In [ ]:
me = obs_df.residual_m.mean()
mae = obs_df.residual_m.abs().mean()
rmse = np.sqrt((obs_df.residual_m**2).mean())
ss_res = (obs_df.residual_m**2).sum()
ss_tot = ((obs_df["observed_m (SYNTHÉTIQUE)"] - obs_df["observed_m (SYNTHÉTIQUE)"].mean())**2).sum()
r2 = 1 - ss_res/ss_tot

print(f"Erreur moyenne (ME) :        {me:.3f} m")
print(f"Erreur absolue moyenne (MAE) : {mae:.3f} m")
print(f"RMSE :                        {rmse:.3f} m")
print(f"R² :                          {r2:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(obs_df["observed_m (SYNTHÉTIQUE)"], obs_df.simulated_m, color="navy")
lims = [obs_df[["observed_m (SYNTHÉTIQUE)","simulated_m"]].min().min()-1,
        obs_df[["observed_m (SYNTHÉTIQUE)","simulated_m"]].max().max()+1]
axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set_xlabel("Charge observée (synthétique) (m)"); axes[0].set_ylabel("Charge simulée (m)")
axes[0].set_title("Observé vs simulé")

axes[1].hist(obs_df.residual_m, bins=10, color="steelblue", edgecolor="k")
axes[1].axvline(0, color="k", lw=1)
axes[1].set_xlabel("Résidu (observé - simulé), m"); axes[1].set_title("Histogramme des résidus")

sc = axes[2].scatter(obs_df.col*delr, Ly-(obs_df.row*delc), c=obs_df.residual_m,
                      cmap="RdBu_r", vmin=-1.5, vmax=1.5, s=60, edgecolor="k")
axes[2].set_xlim(0, Lx); axes[2].set_ylim(0, Ly)
axes[2].set_title("Répartition spatiale des résidus")
plt.colorbar(sc, ax=axes[2], label="Résidu (m)")
plt.tight_layout()
plt.savefig("../figures/stage18_calibration.pdf", dpi=150)
plt.show()